# Progress Tracker Agent — Demo

Reads all phase JSON files from `phase_outputs/` and generates a longitudinal progress report.

**Pipeline:**
```
phase_outputs/*.json → Load → Trend Analysis → gemma3:4b → Progress Report
```

## 1. Setup

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))

from progress_tracker import ProgressTracker, load_phase_jsons, PHASE_OUTPUTS_DIR

print('ProgressTracker loaded ✓')
print(f'Reading from: {PHASE_OUTPUTS_DIR}')

## 2. Inspect Available Phase Files

In [ ]:
json_files = sorted(PHASE_OUTPUTS_DIR.glob('*.json'))
print(f'Found {len(json_files)} phase file(s):\n')
for f in json_files:
    with open(f, encoding='utf-8') as fp:
        d = json.load(fp)
    ps = d.get('phase_summary', {})
    print(f'  • {f.name}')
    print(f'    Patient  : {ps.get("patient_id", "?")}')
    print(f'    Phase    : {ps.get("rehab_phase", "?")}')
    print(f'    Pain     : {ps.get("pain_level", "?")}/10')
    print(f'    Exercises: {[ex["exercise_name"] for ex in d.get("exercises", [])]}')
    print()

## 3. Run Progress Tracker

In [ ]:
tracker = ProgressTracker(
    ollama_model = 'gemma3:4b',
    verbose      = True,
)

# Pass patient_id to filter, or None to analyze all files
result = tracker.run(patient_id=None)

## 4. Print Progress Report

In [ ]:
print('=' * 65)
print('  LONGITUDINAL PROGRESS REPORT')
print('=' * 65)
print(f'  Patient   : {result["patient_id"]}')
print(f'  Condition : {result["condition"]}')
print(f'  Phases    : {result["phases_analyzed"]} ({result["phase_progression"]})')
print()

# ── Trend Summary ────────────────────────────────────────────────────────
pain = result['pain_analysis']
qual = result['quality_analysis']
mist = result['mistake_analysis']

print('  📉 PAIN TREND')
pain_values = ' → '.join(str(v) for v in pain['values'])
print(f'     {pain_values}/10  ({pain["trend"]})')

print('\n  📈 QUALITY TREND')
qual_values = ' → '.join(
    str(v) if v is not None else 'N/A'
    for v in qual['values_per_phase']
)
print(f'     {qual_values}  ({qual["trend"]})')

print('\n  ⚠️  TOP MISTAKES')
for mtype, total in mist['ranked'][:3]:
    flag = ' ← PERSISTENT' if mtype in mist['persistent'] else ''
    print(f'     • {mtype}: {total} occurrences{flag}')

print('\n  🏋️  EXERCISE PROGRESSION')
for line in result['exercise_progression']:
    print(f'     {line}')

print()
print('=' * 65)
print('  FULL REPORT')
print('=' * 65)
print()
for line in result['progress_report'].strip().split('\n'):
    print(f'  {line}')
print()

## 5. Inspect Raw Structured Output

In [ ]:
# Full structured result (excluding long report text for readability)
summary = {k: v for k, v in result.items() if k != 'progress_report'}
print(json.dumps(summary, indent=2, default=str))